<a href="https://colab.research.google.com/github/obrnocsta/atendimento-clientes/blob/main/1-desafio/atendimento_cliente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Parte I - Conexão e Testes

**01 - Instalar**

In [1]:
!pip install -q gspread faker

**02 - Importações e Autenticação (service account)**

In [2]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("CLIENT_SECRETS_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@atendimento-ao-cliente-513822.iam.gserviceaccount.com


**03 - Abrir a Planilha**

In [3]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1K-_cCfzJ0DGdOrdAq8bGEOKfcgMw7jxfi-05GcqtRVA"
NOME_TABELA = "clientes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Atendimento_Cliente | clientes | https://docs.google.com/spreadsheets/d/1K-_cCfzJ0DGdOrdAq8bGEOKfcgMw7jxfi-05GcqtRVA


**04 - Cabeçalho - Tabela Dimensão (clientes)**

In [14]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["cliente_id", "segmento", "canal_preferido", "data_cadastro"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


**05 - 10 linhas de teste - Tabela Dimensão (clientes)**

In [16]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados

    # Intervalo de datas exigido: Entre 01/01/2019 e 01/07/2026
    # Calculamos a diferença de dias a partir de uma data de referência fixa
    from datetime import date, timedelta
    data_inicio = date(2019, 1, 1)
    data_fim = date(2026, 7, 1)
    delta_dias = (data_fim - data_inicio).days

    # Sorteia um número de dias para somar à data inicial
    data_sorteada = data_inicio + timedelta(days=random.randint(0, delta_dias))

    # Definição das categorias e seus respectivos pesos percentuais
    segmentos = ["pessoa_fisica", "pequena_empresa", "grande_empresa"]
    pesos_segmento = [0.75, 0.20, 0.05]

    canais = ["chat", "telefone", "email", "presencial"]
    pesos_canal = [0.35, 0.30, 0.25, 0.10]

    return {
        "cliente_id": i,
        # random.choices retorna uma lista, extraímos o primeiro item com [0]
        "segmento": random.choices(segmentos, weights=pesos_segmento)[0],
        "canal_preferido": random.choices(canais, weights=pesos_canal)[0],
        "data_cadastro": data_sorteada.isoformat(), # Retorna no formato texto YYYY-MM-DD
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]

# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

10 linhas enviadas


**06 - Conferência – Tabela Dimensão (clientes)**

In [17]:
# Lê todos os valores da aba para conferir o conteúdo enviado.
valores_lidos = ws.get_all_values()
# Exibe a quantidade de linhas, confere o cabeçalho e mostra exemplos.
print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 11
Cabeçalho confere: True
[['cliente_id', 'segmento', 'canal_preferido', 'data_cadastro'], ['1', 'pessoa_fisica', 'email', '2026-03-04'], ['2', 'pessoa_fisica', 'email', '2021-09-30']]


## Parte II - Base Completa

**07 - Funções de envio de dados - Planilha**

In [18]:
# Importa recursos para controlar pausas, acessar a planilha e tratar tabelas.
import time
import gspread
import pandas as pd

def df_para_valores(df):
    """DataFrame -> lista de listas com tipos nativos do Python."""

    # Trabalha em uma cópia para não alterar o DataFrame original.
    df = df.copy()
    for c in df.columns:
        # Converte datas para texto em um formato padronizado.
        if pd.api.types.is_datetime64_any_dtype(df[c]):
            df[c] = df[c].dt.strftime("%Y-%m-%d %H:%M:%S")
        # Converte outros valores de data e hora para texto.
        elif df[c].dtype == object:
            df[c] = df[c].map(
                lambda v: v.isoformat() if hasattr(v, "isoformat") else v)

    # Substitui valores ausentes por texto vazio e retorna linhas de valores.
    return df.astype(object).where(df.notna(), "").values.tolist()

def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    """Escreve cabeçalho e linhas numa aba, em lotes, e confere a contagem."""

    # Formata números com ponto como separador de milhar.
    n = lambda x: f"{x:,}".replace(",", ".")
    try:
        # Reutiliza a aba existente e apaga seu conteúdo anterior.
        ws = sh.worksheet(nome_aba)
        ws.clear()
    except gspread.WorksheetNotFound:
        # Cria uma aba caso ela ainda não exista.
        ws = sh.add_worksheet(title=nome_aba, rows=len(df) + 1,
                              cols=len(df.columns))

    # Ajusta o tamanho da aba e grava os nomes das colunas.
    ws.resize(rows=len(df) + 1, cols=len(df.columns))
    ws.update(range_name="A1", values=[list(df.columns)])

    # Converte o DataFrame e envia as linhas em lotes.
    valores = df_para_valores(df)
    for i in range(0, len(valores), lote):
        ws.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {n(min(i + lote, len(valores)))} / {n(len(valores))}")
        time.sleep(pausa)               # respeita o limite de escritas/minuto

    # Confere se a quantidade de linhas gravadas corresponde à esperada.
    lidas = len(ws.col_values(1)) - 1   # lê de volta e confere
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {n(len(df))} linhas na planilha")

def remover_aba_padrao(sh):
    """Apaga a aba vazia que o Google cria junto com a planilha nova."""

    # Localiza e remove a aba padrão somente se houver outra aba disponível.
    abas = sh.worksheets()
    padrao = ("Página1", "Sheet1", "Planilha1")
    for w in abas:
        if w.title in padrao and len(abas) > 1 and not w.get_all_values():
            sh.del_worksheet(w)

**08 - Carregar Bases de dados - Prontas**

In [19]:
# Abre a opção de selecionar arquivos CSV no Colab e envia os arquivos.
from google.colab import files
files.upload()   # selecione os arquivos CSV do seu tema

# Carrega cada CSV em um DataFrame e associa-o ao nome da tabela.
# Altere o nome da tabela de acordo com a sua Base de dados
BASES = {
    "clientes": pd.read_csv("clientes.csv"),
    "chamados": pd.read_csv("chamados.csv"),
}

# Calcula e exibe a quantidade de linhas carregadas.
total = sum(len(d) for d in BASES.values())
print({n: len(d) for n, d in BASES.items()}, "| total:", total)

Saving chamados.csv to chamados.csv
Saving clientes.csv to clientes.csv
{'clientes': 10400, 'chamados': 42000} | total: 52400


**09 - Enviar para a planilha em lotes**

In [20]:
# Envia cada tabela para uma aba e remove a aba padrão, se estiver vazia.
for nome, df in BASES.items():
    enviar(sh, df, nome)        # uma aba por tabela
remover_aba_padrao(sh)

  clientes: 5.000 / 10.400
  clientes: 10.000 / 10.400
  clientes: 10.400 / 10.400
OK clientes: 10.400 linhas na planilha
  chamados: 5.000 / 42.000
  chamados: 10.000 / 42.000
  chamados: 15.000 / 42.000
  chamados: 20.000 / 42.000
  chamados: 25.000 / 42.000
  chamados: 30.000 / 42.000
  chamados: 35.000 / 42.000
  chamados: 40.000 / 42.000
  chamados: 42.000 / 42.000
OK chamados: 42.000 linhas na planilha


**10 - Validar a base na planilha**

In [21]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(d) for d in BASES.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))

# Interrompe a execução se o volume mínimo não for atingido.
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"

# Confirma que todas as verificações passaram.
print("A planilha confere com a base gerada.")

Total de linhas: 52.400
A planilha confere com a base gerada.
